# Volatility Regression 05 — prospective confirmation scoring

Scores the frozen B0 and B0_ATM models from `04/04_frozen_spec.json` on expiries **≥ 2026-10-09**. Nothing is refitted or selected. The previous expiry is loaded only so that lag features at the start of 2026-10-09 are complete; it is never scored.

- **Rule** (VR 011): an asset passes if B0_ATM has lower session log-RMSE than B0 in ≥ 2/3 of counted sessions (≥ 100 comparison rows) **and** lower pooled log-RMSE. The result is confirmed if ≥ 2 of BTC/ETH/SOL pass on the Parkinson target.
- **Binding** at ≥ 20 counted sessions; before that the readout is interim and non-binding.
- Secondary targets and assets are reported and never decisive.

Re-run whenever new sessions are available; the outputs are overwritten each time.

In [1]:
from pathlib import Path
import os, sys, json, subprocess
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import duckdb
from dotenv import load_dotenv

HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents] if (p / "src" / "data_engineering").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Launch from the repository root or one of its subfolders.")
sys.path.insert(0, str(ROOT / "src"))
load_dotenv(ROOT / ".env")
DATA_ROOT = os.getenv("DERIBIT_DATA_ROOT")
if not DATA_ROOT:
    raise RuntimeError("DERIBIT_DATA_ROOT must be set in .env.")
DB_PATH = Path(DATA_ROOT) / "database" / "research.duckdb"
if not DB_PATH.is_file():
    raise FileNotFoundError(DB_PATH)
from data_engineering.modeling import vol_ols_confirm as vc
OUTPUT = ROOT / "notebooks" / "Volatility_Regression" / "05"   # outputs live in a numbered subfolder
OUTPUT.mkdir(parents=True, exist_ok=True)
SPEC_PATH = OUTPUT.parent / "04" / "04_frozen_spec.json"
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(ROOT / "tests"),
                "-p", "test_vol_ols_confirm.py", "-v"], cwd=ROOT, check=True)

CompletedProcess(args=['D:\\Anaconda\\envs\\deribit-data\\python.exe', '-m', 'unittest', 'discover', '-s', 'D:\\Projects\\deribit_market_data\\tests', '-p', 'test_vol_ols_confirm.py', '-v'], returncode=0)

In [2]:
if not SPEC_PATH.exists():
    raise FileNotFoundError("Run and commit notebook 04 first.")
frozen = json.loads(SPEC_PATH.read_text(encoding="utf-8"))
if frozen["registration"] != json.loads(vc.dumps(vc.registration())):
    raise RuntimeError("Module constants differ from the frozen registration. Do not edit them.")
prev = (pd.Timestamp(vc.CONFIRM_FIRST_EXPIRY) - pd.Timedelta(days=1)).strftime("%Y-%m-%d")
with duckdb.connect(str(DB_PATH), read_only=True) as con:
    con.execute("SET TimeZone = 'UTC'")
    df = con.execute("""
        SELECT * FROM time_series_feature_state
        WHERE TRY_CAST(expiry_date AS DATE) >= CAST(? AS DATE)
        ORDER BY underlying, state_time
    """, [prev]).df()
print("Rows loaded (incl. lag-only prior session):", len(df))
print("Confirmation expiries available:",
      sorted(e for e in df["expiry_date"].astype(str).str[:10].unique() if e >= vc.CONFIRM_FIRST_EXPIRY))

Rows loaded (incl. lag-only prior session): 62496
Confirmation expiries available: ['2026-10-09', '2026-10-10', '2026-10-11', '2026-10-12', '2026-10-13', '2026-10-14', '2026-10-15', '2026-10-16', '2026-10-17', '2026-10-18', '2026-10-19', '2026-10-20', '2026-10-21', '2026-10-22', '2026-10-23', '2026-10-24', '2026-10-25', '2026-10-26', '2026-10-27', '2026-10-28', '2026-10-29', '2026-10-30', '2026-10-31', '2026-11-01', '2026-11-02', '2026-11-03', '2026-11-04', '2026-11-05', '2026-11-06', '2026-11-07']


In [3]:
session_frames, verdict_rows = [], []
for asset, by_target in frozen["specs"].items():
    g = df[df["underlying"] == asset]
    if g.empty:
        continue
    meta, X = vc.panel(g)
    for target, spec in by_target.items():
        s = vc.score_asset(meta, X, spec)
        if s.empty:
            verdict_rows.append({"underlying": asset, "target": target, "status": "no_data"})
            continue
        v = vc.asset_verdict(s)
        role = ("primary" if asset in vc.PRIMARY_ASSETS and target == vc.PRIMARY_TARGET
                else "secondary")
        verdict_rows.append({"underlying": asset, "target": target, "role": role,
                             "n_pooled": s.attrs.get("n_pooled"), **v})
        s.insert(0, "target", target)
        s.insert(0, "underlying", asset)
        session_frames.append(s)
sessions_df = pd.concat(session_frames, ignore_index=True) if session_frames else pd.DataFrame()
verdict_df = pd.DataFrame(verdict_rows)
primary = {r["underlying"]: r for r in verdict_rows
           if r.get("role") == "primary"}
for a in vc.PRIMARY_ASSETS:
    primary.setdefault(a, {"status": "no_data"})
overall = vc.overall_verdict(primary)
display(verdict_df.round(4))
print("OVERALL (primary, pre-registered rule):", overall)

,underlying,target,role,n_pooled,counted_sessions,atm_wins,win_share,pooled_B0,pooled_B0_ATM,pooled_persistence,pooled_train_mean,status
0,AVAX,parkinson,secondary,3812,22,11,0.5000,1.5156,1.5155,1.9088,1.7363,fail
1,AVAX,index_abs,secondary,3812,22,3,0.1364,1.1652,1.1694,1.5910,1.2354,fail
2,AVAX,mid_rv,secondary,4614,28,11,0.3929,0.3507,0.3518,0.3762,0.4986,fail
3,BTC,parkinson,primary,5054,29,20,0.6897,0.4872,0.4866,0.5828,0.7128,pass
4,BTC,index_abs,secondary,5054,29,14,0.4828,1.1152,1.1174,1.5212,1.2027,fail
5,BTC,mid_rv,secondary,5567,30,17,0.5667,0.4114,0.4034,0.4390,0.6344,fail
6,ETH,parkinson,primary,5126,29,25,0.8621,0.4522,0.4450,0.5356,0.6633,pass
7,ETH,index_abs,secondary,5126,29,19,0.6552,1.0807,1.0785,1.4743,1.1722,fail
8,ETH,mid_rv,secondary,5668,30,21,0.7000,0.3745,0.3694,0.3850,0.6180,pass
9,HYPE,parkinson,secondary,4559,26,6,0.2308,1.3461,1.5062,1.5771,1.5833,fail


OVERALL (primary, pre-registered rule): confirmed


In [4]:
sessions_df.to_csv(OUTPUT / "05_session_scores.csv", index=False)
verdict_df.to_csv(OUTPUT / "05_verdict.csv", index=False)
(OUTPUT / "05_run_metadata.json").write_text(vc.dumps({
    "scored_at_utc": datetime.now(timezone.utc).isoformat(),
    "frozen_at_utc": frozen["frozen_at_utc"],
    "overall": overall,
    "data_last_state": str(df.state_time.max()),
}), encoding="utf-8")
print("Saved 05_session_scores.csv, 05_verdict.csv, 05_run_metadata.json")

Saved 05_session_scores.csv, 05_verdict.csv, 05_run_metadata.json
